# Préparation Kanakmi Mental Disorders 

**Décisions prises pour ce dataset :**
- Anxiety conservée comme 5e pathologie du projet
- PTSD absent de ce dataset (gap documenté — voir `config.json` en sortie)
- Rôle cible : éducatif (R2)
- Racine Drive unifiée avec Shifaa : `Master_Thesis_RAG_Indexes/`

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import re
from google.colab import drive

print("="*60)
print("PRÉPARATION DU DATASET MENTAL DISORDERS (Kanakmi)")
print("="*60)

drive.mount('/content/drive')


BASE_DRIVE = '/content/drive/MyDrive/Master_Thesis_RAG_Indexes'
OUTPUT_DIR = f"{BASE_DRIVE}/Kanakmi"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 Dossier de sortie : {OUTPUT_DIR}")

PRÉPARATION DU DATASET MENTAL DISORDERS (Kanakmi)
Mounted at /content/drive
📁 Dossier de sortie : /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Kanakmi


In [ ]:
#  Chargement 
print("\n📥 Chargement du dataset Mental Disorders...")

splits = {
    'train': 'data/train-00000-of-00001.parquet',
    'test': 'data/test-00000-of-00001.parquet',
    'val': 'data/val-00000-of-00001.parquet'
}

df_train = pd.read_parquet("hf://datasets/Kanakmi/mental-disorders/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Kanakmi/mental-disorders/" + splits["test"])
df_val = pd.read_parquet("hf://datasets/Kanakmi/mental-disorders/" + splits["val"])

df = pd.concat([df_train, df_test, df_val], ignore_index=True)
print(f"✅ Dataset chargé : {len(df)} exemples totaux")


📥 Chargement du dataset Mental Disorders...
✅ Dataset chargé : 581217 exemples totaux


In [ ]:
# Mapping des labels 
label_mapping = {
    0: "BPD",
    1: "Bipolar",
    2: "Depression",
    3: "Anxiety",
    4: "Schizophrenia",
    5: "Mental_Illness_General"  # exclu
}
df['label_name'] = df['label'].map(label_mapping)

target_labels = [0, 1, 2, 3, 4]
df_filtered = df[df['label'].isin(target_labels)].copy()
print(f"✅ Après filtrage (exclusion Mental_Illness_General) : {len(df_filtered)} exemples")
print(df_filtered['label_name'].value_counts())

✅ Après filtrage (exclusion Mental_Illness_General) : 543056 exemples
label_name
BPD              212826
Anxiety          161629
Depression       121202
Bipolar           35672
Schizophrenia     11727
Name: count, dtype: int64


In [ ]:
# Nettoyage du texte 
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('[deleted]', '').replace('[removed]', '')
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_filtered['text_clean'] = df_filtered['text'].apply(clean_text)
df_filtered = df_filtered[df_filtered['text_clean'].str.len().between(50, 2000)]
df_filtered = df_filtered.drop_duplicates(subset=['text_clean'])
print(f"✅ Après nettoyage : {len(df_filtered)} exemples valides")

✅ Après nettoyage : 470258 exemples valides


In [ ]:
# Équilibrage 
MAX_SAMPLES_PER_CLASS = 3000
df_balanced = df_filtered.groupby('label', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), MAX_SAMPLES_PER_CLASS), random_state=42)
).reset_index(drop=True)
print(f"✅ Dataset équilibré : {len(df_balanced)} exemples")
print(df_balanced['label_name'].value_counts())

✅ Dataset équilibré : 15000 exemples
label_name
BPD              3000
Bipolar          3000
Depression       3000
Anxiety          3000
Schizophrenia    3000
Name: count, dtype: int64


/tmp/ipykernel_358/927410451.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df_filtered.groupby('label', group_keys=False).apply(


In [ ]:
# Schéma de métadonnées UNIFIÉ (commun avec Shifaa) 
# Champs communs : id, text, lang, role_clinical, role_education, content_type, source, disorder

records = []
for idx, row in df_balanced.iterrows():
    records.append({
        'id': f"kanakmi_{idx}",
        'text': row['text_clean'],
        'lang': 'en',
        'role_clinical': False,       # jamais utilisé par R1
        'role_education': True,        # illustration de vécu pour R2
        'content_type': 'patient_testimony',
        'source': 'Kanakmi/mental-disorders',
        'disorder': row['label_name']
    })

df_final = pd.DataFrame(records)

csv_path = f"{OUTPUT_DIR}/mental_disorders_prepared_rag.csv"
df_final.to_csv(csv_path, index=False, encoding='utf-8')
print(f"✅ CSV sauvegardé : {csv_path}")

metadata_path = f"{OUTPUT_DIR}/metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"✅ Métadonnées sauvegardées : {metadata_path}")

✅ CSV sauvegardé : /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Kanakmi/mental_disorders_prepared_rag.csv
✅ Métadonnées sauvegardées : /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Kanakmi/metadata.json


In [ ]:
# Config avec gaps documentés explicitement 
config = {
    'dataset_name': 'Kanakmi/mental-disorders',
    'role_clinical': False,
    'role_education': True,
    'content_type': 'patient_testimony',
    'lang': 'en',
    'disorders_included': ['BPD', 'Bipolar', 'Depression', 'Anxiety', 'Schizophrenia'],
    'disorders_excluded': ['Mental_Illness_General'],
    'known_gaps': {
        'PTSD': 'Absent de ce dataset ',
        'Anxiety': 'Ajoutée au périmètre du projet via ce dataset (5e pathologie)'
    },
    'total_samples': len(df_final),
    'samples_per_disorder': df_final['disorder'].value_counts().to_dict(),
    'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
    'created_date': pd.Timestamp.now().isoformat()
}

config_path = f"{OUTPUT_DIR}/config.json"
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print(f"✅ Configuration sauvegardée : {config_path}")
print("\n🎉 Terminé — schéma unifié avec Shifaa, prêt pour l'embedding.")

✅ Configuration sauvegardée : /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Kanakmi/config.json

🎉 Terminé — schéma unifié avec Shifaa, prêt pour l'embedding.
